# ✈️ Flight Trajectory Analysis — Spatio-Temporal Big Data Analytics
**BDSMA Micro Project · SDG 9: Industry, Innovation & Infrastructure**

**Problem:** Indian airspace handles 4,000+ daily flights. Congested terminal areas, airway detours and
holding patterns waste fuel, add CO₂ and delay passengers. We use big-data tools on ADS-B
*state vectors* (OpenSky Network schema) to find **where & when** airspace is congested, **which routes are
inefficient**, and **which airports are structural hubs** — insights for air-navigation infrastructure planning.

| Stage | Syllabus concept | Tool |
|---|---|---|
| 1 | HDFS storage & file ops | Hadoop 3.3.6 (pseudo-distributed) |
| 2 | MapReduce: spatio-temporal grid aggregation, trajectory reconstruction | Hadoop Streaming (Python) |
| 3 | Distributed processing, window functions | Apache Spark (PySpark) |
| 4 | NoSQL document store + geospatial index | MongoDB (`2dsphere`) |
| 5 | Community detection / structural analytics | Louvain, PageRank, Betweenness |
| 6 | Visualisation | Folium + web dashboard |

> Runtime → *Run all*. Takes ~6–8 min (mostly Hadoop download).

## 0. Setup — Java + Hadoop (single-node HDFS)

In [ ]:
!apt-get -qq install -y openjdk-11-jdk-headless > /dev/null
!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz && mv hadoop-3.3.6 /usr/local/hadoop
!pip -q install pyspark pymongo folium networkx

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop"
os.environ["PATH"] += ":/usr/local/hadoop/bin:/usr/local/hadoop/sbin"
for r in ["HDFS_NAMENODE_USER", "HDFS_DATANODE_USER", "HDFS_SECONDARYNAMENODE_USER"]:
    os.environ[r] = "root"

core = '''<configuration>
 <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>'''
hdfs = '''<configuration>
 <property><name>dfs.replication</name><value>1</value></property>
 <property><name>dfs.permissions.enabled</name><value>false</value></property>
 <property><name>dfs.namenode.name.dir</name><value>/tmp/hdfs/nn</value></property>
 <property><name>dfs.datanode.data.dir</name><value>/tmp/hdfs/dn</value></property>
</configuration>'''
open("/usr/local/hadoop/etc/hadoop/core-site.xml", "w").write(core)
open("/usr/local/hadoop/etc/hadoop/hdfs-site.xml", "w").write(hdfs)
!echo "export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64" >> /usr/local/hadoop/etc/hadoop/hadoop-env.sh

In [ ]:
!hdfs namenode -format -force -nonInteractive 2>&1 | tail -1
!hdfs --daemon start namenode && hdfs --daemon start datanode
!sleep 8 && jps && hdfs dfsadmin -report | head -12

## 1. Dataset
**Primary:** [OpenSky Network historical state vectors](https://opensky-network.org/datasets/states) — real ADS-B
data, one CSV per hour, ~1 GB/day of global traffic. We download one hour and keep the Indian FIR bounding box
(lat 6–37°N, lon 68–98°E).

**Labelled sample:** OpenSky vectors carry no origin/destination, so for route-efficiency & network analysis we also
generate a reproducible 24-hour dataset in the *same schema* for 20 major Indian airports (gravity traffic model,
IST peak hours, airway dog-legs, holding stacks at DEL/BOM).

In [ ]:
%%writefile generate_dataset.py
"""
Generates an ADS-B state-vector dataset in the exact OpenSky Network schema
(https://opensky-network.org/datasets/states) for Indian domestic airspace.

Columns: time, icao24, lat, lon, velocity, heading, vertrate, callsign,
         onground, baroaltitude, geoaltitude, origin, destination

The Colab notebook downloads REAL OpenSky data; this generator gives an
offline, reproducible sample with the same schema (plus origin/destination
labels) so the local pipeline + dashboard work without internet.
"""
import csv, math, random, os

random.seed(42)

AIRPORTS = {  # IATA: (name, lat, lon, weight = relative traffic)
    "DEL": ("Delhi", 28.5562, 77.1000, 10), "BOM": ("Mumbai", 19.0896, 72.8656, 9),
    "BLR": ("Bengaluru", 13.1986, 77.7066, 8), "HYD": ("Hyderabad", 17.2403, 78.4294, 6),
    "MAA": ("Chennai", 12.9941, 80.1709, 6), "CCU": ("Kolkata", 22.6547, 88.4467, 5),
    "AMD": ("Ahmedabad", 23.0772, 72.6347, 4), "PNQ": ("Pune", 18.5821, 73.9197, 4),
    "COK": ("Kochi", 10.1520, 76.4019, 3), "GOI": ("Goa", 15.3808, 73.8314, 3),
    "JAI": ("Jaipur", 26.8242, 75.8122, 2), "LKO": ("Lucknow", 26.7606, 80.8893, 2),
    "GAU": ("Guwahati", 26.1061, 91.5859, 2), "PAT": ("Patna", 25.5913, 85.0880, 2),
    "BBI": ("Bhubaneswar", 20.2444, 85.8178, 2), "IXC": ("Chandigarh", 30.6735, 76.7885, 2),
    "NAG": ("Nagpur", 21.0922, 79.0472, 2), "TRV": ("Trivandrum", 8.4821, 76.9200, 2),
    "SXR": ("Srinagar", 33.9871, 74.7742, 1), "IXB": ("Bagdogra", 26.6812, 88.3286, 1),
}
AIRLINES = ["AIC", "IGO", "VTI", "AKJ", "SEJ", "IAD"]
R = 6371.0


def hav(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def bearing(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    y = math.sin(dl) * math.cos(p2)
    x = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dl)
    return (math.degrees(math.atan2(y, x)) + 360) % 360


def build_path(o, d):
    """Waypoints: origin -> (optional airway dog-leg) -> destination."""
    (_, la1, lo1, _), (_, la2, lo2, _) = AIRPORTS[o], AIRPORTS[d]
    pts = [(la1, lo1)]
    if random.random() < 0.35:  # airway routing / ATC vectoring detour
        f = random.uniform(0.3, 0.7)
        mla, mlo = la1 + (la2 - la1) * f, lo1 + (lo2 - lo1) * f
        off = random.uniform(0.4, 1.6) * random.choice([-1, 1])
        pts.append((mla + off * (lo2 - lo1) / 10, mlo - off * (la2 - la1) / 10))
    pts.append((la2, lo2))
    return pts


def sample_path(pts, n):
    segs = [hav(*pts[i], *pts[i + 1]) for i in range(len(pts) - 1)]
    total, out = sum(segs), []
    for k in range(n):
        s, i = total * k / (n - 1), 0
        while i < len(segs) - 1 and s > segs[i]:
            s -= segs[i]; i += 1
        f = s / segs[i] if segs[i] else 0
        (a, b), (c, e) = pts[i], pts[i + 1]
        out.append((a + (c - a) * f, b + (e - b) * f))
    return out, total


def main():
    codes = list(AIRPORTS)
    weights = [AIRPORTS[c][3] for c in codes]
    base_t = 1719446400  # 2024-06-27 00:00:00 UTC (OpenSky dataset day style)
    out = os.path.join(os.path.dirname(__file__), "flights_states.csv")
    rows = 0
    with open(out, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["time", "icao24", "lat", "lon", "velocity", "heading", "vertrate",
                    "callsign", "onground", "baroaltitude", "geoaltitude", "origin", "destination"])
        for fid in range(900):
            # gravity model: traffic ~ size(dest) / distance^1.3 -> regional structure
            o = random.choices(codes, weights)[0]
            dw = [0 if c == o else AIRPORTS[c][3] / hav(AIRPORTS[o][1], AIRPORTS[o][2],
                  AIRPORTS[c][1], AIRPORTS[c][2]) ** 1.3 for c in codes]
            d = random.choices(codes, dw)[0]
            # IST traffic peaks: morning 06-10, evening 17-21 (UTC = IST-5.5)
            hour_ist = random.choices(range(24), [1,1,1,1,2,4,9,10,9,7,5,5,5,5,5,6,7,9,10,9,7,4,2,1])[0]
            t0 = base_t + int(((hour_ist - 5.5) % 24) * 3600) + random.randint(0, 3599)
            pts = build_path(o, d)
            gc = hav(*pts[0], *pts[-1])
            cruise_v = random.uniform(220, 245)  # m/s
            dur = gc * 1000 / cruise_v * 1.15 + 600
            n = max(20, int(dur / 60))  # one state vector per minute
            path, _ = sample_path(pts, n)
            hold = d in ("DEL", "BOM") and random.random() < 0.25  # congestion holding
            icao = "%06x" % (0x800000 + fid * 37)
            cs = random.choice(AIRLINES) + str(random.randint(100, 9999))
            cruise_alt = random.choice([9450, 10050, 10650, 11250, 11850]) if gc > 400 else 7600
            seq = []
            for k, (la, lo) in enumerate(path):
                f = k / (n - 1)
                if f < 0.15:   alt, vr = cruise_alt * f / 0.15, cruise_alt / (0.15 * dur)
                elif f > 0.85: alt, vr = cruise_alt * (1 - f) / 0.15, -cruise_alt / (0.15 * dur)
                else:          alt, vr = cruise_alt, random.uniform(-0.5, 0.5)
                seq.append([la, lo, alt, vr])
            if hold:  # insert racetrack orbits ~40 km before destination
                la_h, lo_h = seq[-8][0], seq[-8][1]
                orbit = [[la_h + 0.12 * math.sin(a / 6 * math.pi), lo_h + 0.18 * math.cos(a / 6 * math.pi),
                          2400, 0] for a in range(24)]
                seq = seq[:-8] + orbit + seq[-8:]
            for k, (la, lo, alt, vr) in enumerate(seq):
                nxt = seq[min(k + 1, len(seq) - 1)]
                hd = bearing(la, lo, nxt[0], nxt[1]) if k < len(seq) - 1 else 0
                v = 75 if alt < 300 else (140 if alt < 3000 else cruise_v + random.gauss(0, 4))
                la += random.gauss(0, 0.004); lo += random.gauss(0, 0.004)
                ground = alt < 30
                w.writerow([t0 + k * 60, icao, round(la, 5), round(lo, 5), round(v, 1), round(hd, 1),
                            round(vr, 2), cs, ground, round(max(alt, 0), 1),
                            round(max(alt, 0) + 120, 1), o, d])
                rows += 1
    print(f"wrote {rows} state vectors -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
!python generate_dataset.py
import pandas as pd, subprocess
COLS = ["time","icao24","lat","lon","velocity","heading","vertrate","callsign","onground",
        "baroaltitude","geoaltitude","origin","destination"]
try:  # REAL OpenSky hour (2022-06-27 06:00 UTC = Indian morning peak)
    url = "https://s3.opensky-network.org/data-samples/states/2022-06-27/06/states_2022-06-27-06.csv.tar"
    subprocess.run(f"wget -q -O os.tar {url} && tar -xf os.tar", shell=True, check=True)
    raw = pd.read_csv("states_2022-06-27-06.csv.gz")
    india = raw[raw.lat.between(6, 37) & raw.lon.between(68, 98)].dropna(subset=["lat", "lon"])
    india = india.assign(origin="", destination="")[COLS]
    india.to_csv("opensky_india.csv", index=False)
    print("Real OpenSky India rows:", len(india), "| aircraft:", india.icao24.nunique())
except Exception as e:
    print("OpenSky download unavailable (", e, ") -> using labelled sample only")
    pd.read_csv("flights_states.csv").to_csv("opensky_india.csv", index=False)
pd.read_csv("flights_states.csv").head()

## 2. HDFS operations (Experiment 1)

In [ ]:
!hdfs dfs -mkdir -p /flights/raw /flights/archive
!hdfs dfs -put -f flights_states.csv /flights/raw/          # copy local -> HDFS
!hdfs dfs -put -f opensky_india.csv /flights/raw/
!hdfs dfs -ls -h /flights/raw                                # list
!hdfs dfs -cat /flights/raw/flights_states.csv | head -5     # display
!hdfs dfs -du -h /flights                                    # disk usage
!hdfs dfs -cp /flights/raw/opensky_india.csv /flights/archive/   # copy inside HDFS
!hdfs dfs -mv /flights/archive/opensky_india.csv /flights/archive/opensky_backup.csv  # move/rename
!hdfs dfs -get -f /flights/archive/opensky_backup.csv ./from_hdfs.csv   # HDFS -> local
!hdfs dfs -rm /flights/archive/opensky_backup.csv            # delete
!hdfs fsck /flights/raw/flights_states.csv -files -blocks | grep -E "blocks|Status"

## 3. MapReduce with Hadoop Streaming (Experiment 3)
**Job 1 — Spatio-temporal density grid.** Map: each ADS-B point → key `(lat_cell, lon_cell, hour, altitude_band)`.
Reduce: sum. This is a *spatio-temporal histogram* — the basis for hotspot detection.

**Job 2 — Trajectory reconstruction.** Map: group by `icao24|callsign`. Reduce: sort points by time, compute flown
distance vs great-circle distance → **route inefficiency %**.

In [ ]:
%%writefile mapper_grid.py
#!/usr/bin/env python3
"""MapReduce Job 1 (Hadoop Streaming) - spatio-temporal airspace density.
Emits key = <lat_cell>,<lon_cell>,<hour_utc>,<altitude_band>  value = 1
Grid = 0.5 deg cells; altitude bands: GND / LOW(<3km) / MID(3-8km) / HIGH(>8km)."""
import sys

for line in sys.stdin:
    p = line.strip().split(",")
    if not p or p[0] == "time" or len(p) < 10:
        continue
    try:
        t, lat, lon, alt = int(float(p[0])), float(p[2]), float(p[3]), float(p[9] or 0)
    except ValueError:
        continue
    band = "GND" if alt < 30 else "LOW" if alt < 3000 else "MID" if alt < 8000 else "HIGH"
    print(f"{int(lat // 0.5)},{int(lon // 0.5)},{(t // 3600) % 24},{band}\t1")

In [ ]:
%%writefile reducer_sum.py
#!/usr/bin/env python3
"""Generic sum reducer (input sorted by key, as Hadoop guarantees)."""
import sys

cur, total = None, 0
for line in sys.stdin:
    key, _, val = line.rstrip("\n").partition("\t")
    if key != cur:
        if cur is not None:
            print(f"{cur}\t{total}")
        cur, total = key, 0
    total += int(val)
if cur is not None:
    print(f"{cur}\t{total}")

In [ ]:
%%writefile mapper_route.py
#!/usr/bin/env python3
"""MapReduce Job 2 - per-flight trajectory reconstruction (key = icao24|callsign).
Emits flight_key \t time,lat,lon,alt,origin,destination so the reducer receives
all points of one flight together (secondary sort on time done in reducer)."""
import sys

for line in sys.stdin:
    p = line.strip().split(",")
    if not p or p[0] == "time" or len(p) < 13:
        continue
    print(f"{p[1]}|{p[7]}\t{p[0]},{p[2]},{p[3]},{p[9]},{p[11]},{p[12]}")

In [ ]:
%%writefile reducer_route.py
#!/usr/bin/env python3
"""MapReduce Job 2 reducer - computes, per flight: flown distance, great-circle
distance, route inefficiency %, duration, max altitude.
Output: flight \t origin,destination,flown_km,gc_km,ineff_pct,dur_min,max_alt"""
import sys, math


def hav(a, b, c, d):
    p1, p2 = math.radians(a), math.radians(c)
    x = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(d - b) / 2) ** 2
    return 12742 * math.asin(math.sqrt(x))


def emit(key, pts):
    pts.sort()
    flown = sum(hav(pts[i][1], pts[i][2], pts[i + 1][1], pts[i + 1][2]) for i in range(len(pts) - 1))
    gc = hav(pts[0][1], pts[0][2], pts[-1][1], pts[-1][2]) or 1
    dur = (pts[-1][0] - pts[0][0]) / 60
    print(f"{key}\t{pts[0][4]},{pts[0][5]},{flown:.1f},{gc:.1f},{(flown / gc - 1) * 100:.2f},"
          f"{dur:.0f},{max(p[3] for p in pts):.0f}")


cur, pts = None, []
for line in sys.stdin:
    key, _, v = line.rstrip("\n").partition("\t")
    t, la, lo, alt, o, d = v.split(",")
    if key != cur and cur is not None:
        emit(cur, pts); pts = []
    cur = key
    pts.append((int(float(t)), float(la), float(lo), float(alt or 0), o, d))
if cur is not None:
    emit(cur, pts)

In [ ]:
STREAM = "/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"
!hdfs dfs -rm -r -f /flights/out_grid /flights/out_route
!hadoop jar {STREAM} -files mapper_grid.py,reducer_sum.py -mapper "python3 mapper_grid.py" \
   -reducer "python3 reducer_sum.py" -input /flights/raw/flights_states.csv -output /flights/out_grid 2>&1 | tail -3
!hadoop jar {STREAM} -files mapper_route.py,reducer_route.py -mapper "python3 mapper_route.py" \
   -reducer "python3 reducer_route.py" -input /flights/raw/flights_states.csv -output /flights/out_route 2>&1 | tail -3

In [ ]:
print("Top spatio-temporal cells (lat_cell, lon_cell, hour_UTC, band -> count)")
!hdfs dfs -cat /flights/out_grid/part-* | sort -t$'\t' -k2 -nr | head -10
print("\nLeast efficient flights (flight -> origin,dest,flown_km,gc_km,ineff%,dur,max_alt)")
!hdfs dfs -cat /flights/out_route/part-* | sort -t, -k5 -nr | head -10

## 4. Distributed processing with Spark (reads directly from HDFS)

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.appName("FlightTrajectory").getOrCreate()
sdf = spark.read.csv("hdfs://localhost:9000/flights/raw/flights_states.csv", header=True, inferSchema=True)
sdf = sdf.withColumn("hour_ist", ((F.col("time") % 86400) / 3600 + 5.5).cast("int") % 24)
print("partitions:", sdf.rdd.getNumPartitions(), "rows:", sdf.count())
sdf.createOrReplaceTempView("sv")
spark.sql('''SELECT hour_ist, COUNT(DISTINCT icao24) AS active_aircraft, COUNT(*) AS points
             FROM sv GROUP BY hour_ist ORDER BY hour_ist''').show(24)

In [ ]:
# Window functions: per-flight point-to-point turn rate & climb phase segmentation
w = Window.partitionBy("icao24").orderBy("time")
seg = (sdf.withColumn("dh", (F.col("heading") - F.lag("heading").over(w) + 540) % 360 - 180)
          .withColumn("phase", F.when(F.col("baroaltitude") < 30, "Ground")
                                .when(F.col("vertrate") > 1, "Climb")
                                .when(F.col("vertrate") < -1, "Descent").otherwise("Cruise")))
seg.groupBy("phase").count().show()
# holding detection: >= 540 deg net turn below 3000 m
holds = (seg.filter("baroaltitude < 3000").groupBy("icao24", "destination")
            .agg(F.abs(F.sum("dh")).alias("net_turn")).filter("net_turn >= 540"))
holds.groupBy("destination").count().orderBy(F.desc("count")).show()

In [ ]:
# Hotspots: 0.5 deg grid, airborne only
spark.sql('''SELECT FLOOR(lat/0.5)*0.5+0.25 AS clat, FLOOR(lon/0.5)*0.5+0.25 AS clon, COUNT(*) AS n
             FROM sv WHERE baroaltitude > 30 GROUP BY 1,2 ORDER BY n DESC LIMIT 10''').show()

## 5. NoSQL — MongoDB document store with geospatial index (Experiment 2)

In [ ]:
!wget -q https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-7.0.14.tgz
!tar -xzf mongodb-linux-x86_64-ubuntu2204-7.0.14.tgz && mkdir -p /data/db
!mongodb-linux-x86_64-ubuntu2204-7.0.14/bin/mongod --dbpath /data/db --fork --logpath /tmp/mongod.log

In [ ]:
from pymongo import MongoClient, GEOSPHERE
db = MongoClient("mongodb://localhost:27017").airspace
db.trajectories.drop()
df = pd.read_csv("flights_states.csv")
docs = []
for icao, g in df.sort_values("time").groupby("icao24"):
    docs.append({"_id": icao, "callsign": g.callsign.iloc[0], "origin": g.origin.iloc[0],
                 "destination": g.destination.iloc[0], "start": int(g.time.min()), "end": int(g.time.max()),
                 "max_alt": float(g.baroaltitude.max()),
                 "path": {"type": "LineString", "coordinates": g[["lon", "lat"]].values.tolist()}})
db.trajectories.insert_many(docs)
db.trajectories.create_index([("path", GEOSPHERE)])
db.trajectories.create_index([("origin", 1), ("destination", 1)])
print(db.trajectories.count_documents({}), "trajectory documents")

In [ ]:
# Spatial query: which flights passed within 50 km of Nagpur (central-India airway crossroads)?
q = {"path": {"$near": {"$geometry": {"type": "Point", "coordinates": [79.0472, 21.0922]}, "$maxDistance": 50000}}}
print("Flights within 50 km of NAG:", len(list(db.trajectories.find(q, {"_id": 1}))))
# Spatio-temporal: flights intersecting Mumbai TMA polygon during evening peak (17-21 IST)
tma = {"type": "Polygon", "coordinates": [[[72.4,18.7],[73.4,18.7],[73.4,19.5],[72.4,19.5],[72.4,18.7]]]}
base = 1719446400
print("Mumbai TMA, 17-21 IST:", db.trajectories.count_documents(
    {"path": {"$geoIntersects": {"$geometry": tma}}, "start": {"$lt": base + 15.5*3600}, "end": {"$gt": base + 11.5*3600}}))
# Aggregation pipeline: busiest routes
for r in db.trajectories.aggregate([{"$group": {"_id": {"o": "$origin", "d": "$destination"}, "n": {"$sum": 1}}},
                                    {"$sort": {"n": -1}}, {"$limit": 5}]): print(r)

## 6. Structural analytics — airport network, Louvain communities (Experiments 6 & 7)

In [ ]:
import networkx as nx
G = nx.Graph()
for (o, d), n in df.groupby(["origin", "destination"]).icao24.nunique().items():
    G.add_edge(o, d, weight=G.get_edge_data(o, d, {"weight": 0})["weight"] + n)
comms = nx.community.louvain_communities(G, weight="weight", seed=7)
print("Modularity:", round(nx.community.modularity(G, comms, weight="weight"), 3))
for i, c in enumerate(comms): print(f"Community {i}: {sorted(c)}")
pr, bt = nx.pagerank(G, weight="weight"), nx.betweenness_centrality(G)
print(pd.DataFrame({"pagerank": pr, "betweenness": bt}).sort_values("pagerank", ascending=False).head(8))

## 7. Visualisation

In [ ]:
import folium
from folium.plugins import HeatMap
m = folium.Map(location=[22, 80], zoom_start=5, tiles="CartoDB dark_matter")
HeatMap(df[df.baroaltitude > 30][["lat", "lon"]].sample(15000).values.tolist(), radius=6, blur=8).add_to(m)
for icao, g in list(df.groupby("icao24"))[:80]:
    folium.PolyLine(g[["lat", "lon"]].values.tolist(), weight=1, color="#4cc9f0", opacity=.5).add_to(m)
m

## 8. Insights → SDG 9
* **Congestion hotspots** cluster around DEL & BOM terminal areas during IST 06–10 and 17–21 peaks → case for
  additional runway capacity / parallel approach procedures.
* **Holding patterns** at DEL/BOM waste fuel; spatio-temporal detection lets ANSPs quantify delay cost.
* **Route inefficiency** (flown vs great-circle) quantifies the benefit of *Free Route Airspace* / GAGAN-enabled
  direct routings → lower fuel burn and CO₂ (SDG 9 + SDG 13).
* **Louvain communities** reveal regional sub-networks (North/West/South/East) and **PageRank/Betweenness**
  identify hubs where infrastructure investment has the largest network effect.

Run `analytics/pipeline.py` to regenerate `web/data/dashboard.json` for the interactive dashboard.

In [ ]:
!hdfs --daemon stop datanode && hdfs --daemon stop namenode